# DWI Map Viewer — Orthogonal Slice Images

Generates 3-panel orthoviews (sagittal / coronal / axial middle slices) for every NIfTI map in the results folder, mirroring the folder structure under `results/figures/`.

**Covers:**
- HCP: dtifit maps, NODDI maps, DTI diff maps, NODDI diff maps
- Lab: dtifit maps, NODDI maps, DTI diff maps, NODDI diff maps

**Colormaps** (per-metric, same for base maps and |diff| maps):
- FA → `hot` | MD → `turbo` | NDI → `plasma` | ODI → `viridis` | FWF → `cividis`
- Background (zero voxels) → black via `set_under`

**Scaling:**
- Base maps: auto-scaled to 1st–99th percentile of brain voxels
- |Diff| maps: fixed range from `METRIC_RANGE` (same scale as base metric) so error magnitude is directly comparable by eye; reference maps (zero diff) are skipped automatically


In [1]:
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — faster for batch saving
import matplotlib.pyplot as plt
from pathlib import Path
import re

# ── Paths ──────────────────────────────────────────────────────────────────
RESULTS   = Path('results')
FIG_ROOT  = RESULTS / 'figures'
FIG_ROOT.mkdir(parents=True, exist_ok=True)

# ── Colormap lookup by metric keyword ─────────────────────────────────────
CMAP_MAP = {
    'FA':  'hot',
    'MD':  'turbo',
    'NDI': 'plasma',
    'ODI': 'viridis',
    'FWF': 'cividis',
}
METRICS_OF_INTEREST = {'FA', 'MD', 'NDI', 'ODI', 'FWF'}

# ── Fixed display ranges per metric (used for both base maps and diff maps)
# Diff maps use these same ranges so error magnitude is visually comparable
# to the original metric scale.  Adjust MD if your data uses different units.
METRIC_RANGE = {
    'FA':  (0.0, 1.0),
    'MD':  (0.0, 3e-3),   # mm²/s  (FSL default units)
    'NDI': (0.0, 1.0),
    'ODI': (0.0, 1.0),
    'FWF': (0.0, 1.0),
}

print(f'NiBabel: {nib.__version__}')
print(f'Output root: {FIG_ROOT.resolve()}')


NiBabel: 5.3.3
Output root: D:\UofR\03 Uddin Lab\Codes_Data\results\figures


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# Helper functions
# ══════════════════════════════════════════════════════════════════════════

def get_metric(filename):
    """Detect metric name from filename, e.g. 'dti_FA.nii.gz' → 'FA'."""
    for m in ('FA', 'MD', 'NDI', 'ODI', 'FWF'):
        if m in filename:
            return m
    return None

def is_diff(filename):
    return filename.startswith('diff_')

def parse_title(nii_path, dataset):
    fname = nii_path.name.replace('.nii.gz', '')
    metric = get_metric(nii_path.name)
    metric_str = metric if metric else '?'

    b_match = re.search(r'b(\d{3,4})', str(nii_path))
    n_match = re.search(r'n(\d{2,3})', str(nii_path))
    if not n_match:
        n_match = re.search(r'(\d{2,3})dir', str(nii_path))

    b_str = f'b={b_match.group(1)}' if b_match else ''
    n_str = f'n={n_match.group(1)} dirs' if n_match else ''
    proto = ', '.join(filter(None, [b_str, n_str]))

    parts = str(nii_path).replace('\\', '/')
    if 'diff_maps' in parts:
        model = 'DTI' if 'dtifit' in parts else 'NODDI'
        return f'{dataset} | \u0394{metric_str} ({model}) | {proto}'
    elif 'dtifit' in parts:
        return f'{dataset} | DTI {metric_str} | {proto}'
    elif 'noddi' in parts:
        return f'{dataset} | NODDI {metric_str} | {proto}'
    else:
        return f'{dataset} | {fname}'


def _crop_center(slc, zoom):
    """Crop a 2D slice to 1/zoom of its size, centered, to zoom into the brain."""
    if zoom <= 1.0:
        return slc
    h, w = slc.shape
    nh, nw = max(1, int(h / zoom)), max(1, int(w / zoom))
    r0, c0 = (h - nh) // 2, (w - nw) // 2
    return slc[r0:r0 + nh, c0:c0 + nw]


def save_ortho(nii_path, title, out_png, diff=False, metric=None, zoom=1.0):
    """
    Save a 3-panel orthoview (sagittal / coronal / axial) PNG.
    Returns True if saved, False if skipped.

    Base maps  : per-metric colormap, black background, auto vmin/vmax.
    Diff maps  : RdBu_r, signed, symmetric ±METRIC_RANGE scale, pure white background.
    """
    img  = nib.load(nii_path)
    data = np.squeeze(img.get_fdata())
    if data.ndim != 3:
        print(f'  SKIP (not 3D): {nii_path.name}')
        return False

    sx, sy, sz = data.shape

    # ── Colour limits & style ─────────────────────────────────────────────
    if diff:
        if np.max(np.abs(data)) == 0:
            print(f'  SKIP (ref map, zero diff): {nii_path.name}')
            return False
        _, half = METRIC_RANGE.get(metric, (0.0, 1.0))
        vmin, vmax = -half, half
        # Voxels with |diff| < 1e-9 (background / no change) → NaN → pure white
        data = np.where(np.abs(data) < 1e-9, np.nan, data)
        cmap = plt.get_cmap('RdBu_r').copy()
        cmap.set_bad('white')   # NaN renders as pure white
        fig_bg     = 'white'
        text_color = 'black'
        cbar_color = 'black'
    else:
        flat    = data.ravel()
        flat    = flat[np.isfinite(flat)]
        nonzero = flat[flat > 0]
        if len(nonzero) == 0:
            nonzero = np.array([1e-9, 1.0])
        vmin = 1e-9
        vmax = np.percentile(nonzero, 99)
        cmap = plt.get_cmap(CMAP_MAP.get(metric, 'gray') if metric else 'gray').copy()
        cmap.set_under('black')
        fig_bg     = 'black'
        text_color = 'white'
        cbar_color = 'white'

    # ── Slices (crop + orient) ────────────────────────────────────────────
    sag = _crop_center(data[sx // 2, :, :].T, zoom)   # S-top, PA left-right
    cor = _crop_center(data[:, sy // 2, :].T, zoom)   # S-top, LR left-right
    axi = _crop_center(data[:, :, sz // 2].T, zoom)   # A-top, LR left-right

    # ── Figure ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5),
                             facecolor=fig_bg,
                             gridspec_kw={'wspace': 0.02})
    # Explicitly set figure patch so padding/wspace area matches fig_bg
    fig.patch.set_facecolor(fig_bg)
    fig.suptitle(title, color=text_color, fontsize=11, y=0.98)

    im = None
    for ax, slc in zip(axes, [sag, cor, axi]):
        im = ax.imshow(slc, cmap=cmap, vmin=vmin, vmax=vmax,
                       origin='lower', aspect='equal', interpolation='bilinear')
        ax.axis('off')
        ax.patch.set_facecolor(fig_bg)   # axes patch = same as figure

    # Single colorbar on the right
    cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.80, aspect=25, pad=0.02)
    cbar.ax.yaxis.set_tick_params(color=cbar_color, labelsize=8)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=cbar_color)
    cbar.outline.set_edgecolor(cbar_color)

    out_png.parent.mkdir(parents=True, exist_ok=True)
    # edgecolor='none' prevents a thin border that can look off-white
    plt.savefig(out_png, dpi=200, bbox_inches='tight',
                facecolor=fig_bg, edgecolor='none')
    plt.close(fig)
    return True


def process_directory(scan_dir, fig_base_dir, dataset, diff=False, zoom=1.0):
    """Walk scan_dir for *.nii.gz and save ortho PNGs mirroring the folder structure."""
    all_nii   = sorted(scan_dir.rglob('*.nii.gz'))
    nii_files = [f for f in all_nii if get_metric(f.name) in METRICS_OF_INTEREST]
    if not nii_files:
        print(f'  No maps of interest found in {scan_dir}')
        return 0

    count = 0
    for nii in nii_files:
        rel     = nii.relative_to(scan_dir)
        out_png = fig_base_dir / rel.with_suffix('').with_suffix('.png')
        metric  = get_metric(nii.name)
        d       = diff or is_diff(nii.name)
        title   = parse_title(nii, dataset)

        saved = save_ortho(nii, title, out_png, diff=d, metric=metric, zoom=zoom)
        if saved:
            print(f'  saved: {out_png.relative_to(FIG_ROOT)}')
            count += 1
    return count


print('Helper functions defined.')

Helper functions defined.


## HCP — DTI Maps

In [3]:
scan_dir = RESULTS / 'HCP' / 'dtifit'
fig_dir  = FIG_ROOT / 'HCP' / 'dtifit'
print(f'Processing: {scan_dir}')
n = process_directory(scan_dir, fig_dir, dataset='HCP')
print(f'\nHCP DTI: {n} images saved.')


Processing: results\HCP\dtifit
  saved: HCP\dtifit\b1000_n20\dti_FA.png
  saved: HCP\dtifit\b1000_n20\dti_MD.png
  saved: HCP\dtifit\b1000_n30\dti_FA.png
  saved: HCP\dtifit\b1000_n30\dti_MD.png
  saved: HCP\dtifit\b1000_n45\dti_FA.png
  saved: HCP\dtifit\b1000_n45\dti_MD.png
  saved: HCP\dtifit\b1000_n60\dti_FA.png
  saved: HCP\dtifit\b1000_n60\dti_MD.png
  saved: HCP\dtifit\b1000_n75\dti_FA.png
  saved: HCP\dtifit\b1000_n75\dti_MD.png
  saved: HCP\dtifit\b1000_n90\dti_FA.png
  saved: HCP\dtifit\b1000_n90\dti_MD.png
  saved: HCP\dtifit\b2000_n20\dti_FA.png
  saved: HCP\dtifit\b2000_n20\dti_MD.png
  saved: HCP\dtifit\b2000_n30\dti_FA.png
  saved: HCP\dtifit\b2000_n30\dti_MD.png
  saved: HCP\dtifit\b2000_n45\dti_FA.png
  saved: HCP\dtifit\b2000_n45\dti_MD.png
  saved: HCP\dtifit\b2000_n60\dti_FA.png
  saved: HCP\dtifit\b2000_n60\dti_MD.png
  saved: HCP\dtifit\b2000_n75\dti_FA.png
  saved: HCP\dtifit\b2000_n75\dti_MD.png
  saved: HCP\dtifit\b2000_n90\dti_FA.png
  saved: HCP\dtifit\b2000_

## HCP — NODDI Maps

In [4]:
scan_dir = RESULTS / 'HCP' / 'noddi'
fig_dir  = FIG_ROOT / 'HCP' / 'noddi'
print(f'Processing: {scan_dir}')
n = process_directory(scan_dir, fig_dir, dataset='HCP')
print(f'\nHCP NODDI: {n} images saved.')


Processing: results\HCP\noddi
  saved: HCP\noddi\noddi_b1000_n20_936706\fit_FWF.png
  saved: HCP\noddi\noddi_b1000_n20_936706\fit_NDI.png
  saved: HCP\noddi\noddi_b1000_n20_936706\fit_ODI.png
  saved: HCP\noddi\noddi_b1000_n30_936706\fit_FWF.png
  saved: HCP\noddi\noddi_b1000_n30_936706\fit_NDI.png
  saved: HCP\noddi\noddi_b1000_n30_936706\fit_ODI.png
  saved: HCP\noddi\noddi_b1000_n45_936706\fit_FWF.png
  saved: HCP\noddi\noddi_b1000_n45_936706\fit_NDI.png
  saved: HCP\noddi\noddi_b1000_n45_936706\fit_ODI.png
  saved: HCP\noddi\noddi_b1000_n60_936706\fit_FWF.png
  saved: HCP\noddi\noddi_b1000_n60_936706\fit_NDI.png
  saved: HCP\noddi\noddi_b1000_n60_936706\fit_ODI.png
  saved: HCP\noddi\noddi_b1000_n75_936706\fit_FWF.png
  saved: HCP\noddi\noddi_b1000_n75_936706\fit_NDI.png
  saved: HCP\noddi\noddi_b1000_n75_936706\fit_ODI.png
  saved: HCP\noddi\noddi_b1000_n90_936706\fit_FWF.png
  saved: HCP\noddi\noddi_b1000_n90_936706\fit_NDI.png
  saved: HCP\noddi\noddi_b1000_n90_936706\fit_ODI.pn

## HCP — Difference Maps (DTI + NODDI)

In [5]:
for subtype in ('dtifit', 'noddi'):
    scan_dir = RESULTS / 'HCP' / 'analysis' / 'diff_maps' / subtype
    fig_dir  = FIG_ROOT / 'HCP' / 'analysis' / 'diff_maps' / subtype
    print(f'Processing: {scan_dir}')
    n = process_directory(scan_dir, fig_dir, dataset='HCP', diff=True)
    print(f'  → {n} images saved.\n')


Processing: results\HCP\analysis\diff_maps\dtifit
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b1000_n20.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b1000_n30.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b1000_n45.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b1000_n60.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b1000_n75.png
  SKIP (ref map, zero diff): diff_FA_b1000_n90.nii.gz
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b2000_n20.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b2000_n30.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b2000_n45.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b2000_n60.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b2000_n75.png
  SKIP (ref map, zero diff): diff_FA_b2000_n90.nii.gz
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b3000_n20.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b3000_n30.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b3000_n45.png
  saved: HCP\analysis\diff_maps\dtifit\diff_FA_b3

## Lab — DTI Maps

In [6]:
scan_dir = RESULTS / 'Lab' / 'dtifit'
fig_dir  = FIG_ROOT / 'Lab' / 'dtifit'
print(f'Processing: {scan_dir}')
n = process_directory(scan_dir, fig_dir, dataset='Lab', zoom=1.1)
print(f'\nLab DTI: {n} images saved.')


Processing: results\Lab\dtifit
  saved: Lab\dtifit\b1000_20dir\dti_FA.png
  saved: Lab\dtifit\b1000_20dir\dti_MD.png
  saved: Lab\dtifit\b1000_30dir\dti_FA.png
  saved: Lab\dtifit\b1000_30dir\dti_MD.png
  saved: Lab\dtifit\b1000_45dir\dti_FA.png
  saved: Lab\dtifit\b1000_45dir\dti_MD.png
  saved: Lab\dtifit\b1000_60dir\dti_FA.png
  saved: Lab\dtifit\b1000_60dir\dti_MD.png
  saved: Lab\dtifit\b1000_90dir\dti_FA.png
  saved: Lab\dtifit\b1000_90dir\dti_MD.png
  saved: Lab\dtifit\b2000_20dir\dti_FA.png
  saved: Lab\dtifit\b2000_20dir\dti_MD.png
  saved: Lab\dtifit\b2000_30dir\dti_FA.png
  saved: Lab\dtifit\b2000_30dir\dti_MD.png
  saved: Lab\dtifit\b2000_45dir\dti_FA.png
  saved: Lab\dtifit\b2000_45dir\dti_MD.png
  saved: Lab\dtifit\b2000_60dir\dti_FA.png
  saved: Lab\dtifit\b2000_60dir\dti_MD.png
  saved: Lab\dtifit\b2000_90dir\dti_FA.png
  saved: Lab\dtifit\b2000_90dir\dti_MD.png
  saved: Lab\dtifit\b3000_20dir\dti_FA.png
  saved: Lab\dtifit\b3000_20dir\dti_MD.png
  saved: Lab\dtifit\b30

## Lab — NODDI Maps

In [7]:
scan_dir = RESULTS / 'Lab' / 'noddi'
fig_dir  = FIG_ROOT / 'Lab' / 'noddi'
print(f'Processing: {scan_dir}')
n = process_directory(scan_dir, fig_dir, dataset='Lab', zoom=1.1)
print(f'\nLab NODDI: {n} images saved.')


Processing: results\Lab\noddi
  saved: Lab\noddi\noddi_b1000_n20_962438\fit_FWF.png
  saved: Lab\noddi\noddi_b1000_n20_962438\fit_NDI.png
  saved: Lab\noddi\noddi_b1000_n20_962438\fit_ODI.png
  saved: Lab\noddi\noddi_b1000_n30_962438\fit_FWF.png
  saved: Lab\noddi\noddi_b1000_n30_962438\fit_NDI.png
  saved: Lab\noddi\noddi_b1000_n30_962438\fit_ODI.png
  saved: Lab\noddi\noddi_b1000_n45_962438\fit_FWF.png
  saved: Lab\noddi\noddi_b1000_n45_962438\fit_NDI.png
  saved: Lab\noddi\noddi_b1000_n45_962438\fit_ODI.png
  saved: Lab\noddi\noddi_b1000_n60_962438\fit_FWF.png
  saved: Lab\noddi\noddi_b1000_n60_962438\fit_NDI.png
  saved: Lab\noddi\noddi_b1000_n60_962438\fit_ODI.png
  saved: Lab\noddi\noddi_b1000_n90_962438\fit_FWF.png
  saved: Lab\noddi\noddi_b1000_n90_962438\fit_NDI.png
  saved: Lab\noddi\noddi_b1000_n90_962438\fit_ODI.png
  saved: Lab\noddi\noddi_b2000_n20_962438\fit_FWF.png
  saved: Lab\noddi\noddi_b2000_n20_962438\fit_NDI.png
  saved: Lab\noddi\noddi_b2000_n20_962438\fit_ODI.pn

## Lab — Difference Maps (DTI + NODDI)

In [8]:
for subtype in ('dtifit', 'noddi'):
    scan_dir = RESULTS / 'Lab' / 'analysis' / 'diff_maps' / subtype
    fig_dir  = FIG_ROOT / 'Lab' / 'analysis' / 'diff_maps' / subtype
    print(f'Processing: {scan_dir}')
    n = process_directory(scan_dir, fig_dir, dataset='Lab', diff=True, zoom=1.1)
    print(f'  → {n} images saved.\n')


Processing: results\Lab\analysis\diff_maps\dtifit
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b1000_n20.png
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b1000_n30.png
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b1000_n45.png
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b1000_n60.png
  SKIP (ref map, zero diff): diff_FA_b1000_n90.nii.gz
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b2000_n20.png
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b2000_n30.png
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b2000_n45.png
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b2000_n60.png
  SKIP (ref map, zero diff): diff_FA_b2000_n90.nii.gz
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b3000_n20.png
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b3000_n30.png
  saved: Lab\analysis\diff_maps\dtifit\diff_FA_b3000_n45.png
  SKIP (ref map, zero diff): diff_FA_b3000_n60.nii.gz
  saved: Lab\analysis\diff_maps\dtifit\diff_MD_b1000_n20.png
  saved: Lab\analysis\diff_maps\dtifit\diff_MD_b1000_n30

In [9]:
# ── Final summary ──────────────────────────────────────────────────────────
all_pngs = sorted(FIG_ROOT.rglob('*.png'))
print(f'Total PNG images saved: {len(all_pngs)}')
print(f'Location: {FIG_ROOT.resolve()}\n')

# Count by section
for section in ('HCP/dtifit', 'HCP/noddi', 'HCP/analysis',
                'Lab/dtifit', 'Lab/noddi', 'Lab/analysis'):
    count = len([p for p in all_pngs if section.replace('/', str(Path('/'))) in str(p) or section in str(p).replace('\\', '/')])
    print(f'  {section}: {count} images')

Total PNG images saved: 305
Location: D:\UofR\03 Uddin Lab\Codes_Data\results\figures

  HCP/dtifit: 38 images
  HCP/noddi: 57 images
  HCP/analysis: 75 images
  Lab/dtifit: 30 images
  Lab/noddi: 45 images
  Lab/analysis: 55 images
